# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `byeon120/내 데이터` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `byeon120/my-brain-v1` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0005 · steps 40 · seq 1024 · linear · 양자화 q4_k_m (데이터 4개)


In [ ]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "eyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiIyMDI27JeQIOuMgO2VtCDslYzroKTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IlvsmrTsmIEgMjAyNi0wNi0yMyDsgqzsnbTtgbQjMV0g7JiB7IiZOiA6IO2VhOyImCDsnbjtlITrnbwg7J6s7KCV67mEIOuwjyDstIjquLAg7KeA7ZGcIOyImOynkSAo7Jik64qY7JeF66y0XzIwMjYtMDYtMjMv7Jik64qY67iM66as7ZWRLm1kKSAvIOyYgeyImTogKEFjdGlvbikqKiDqsrDsoJwg7Iuc7Iqk7YWc6rO8IFlvdVR1YmUg7LGE64SQ7J2YIOy1nOyLoCBBUEkg7YKk66W8IOyerOyduOymne2VmOqzoCwg7Jew6rKwIOyDge2DnCDtmZXsnbjsnYQg7JyE7ZWcICjsmKTripjsl4XrrLRfMjAyNi0wNi0yMy/smKTripjruIzrpqztlZEubWQpIC8g7JiB7IiZOiA6IOuNsOydtO2EsCDquLDrsJgg7ISc67mE7IqkIOq4sO2ajCDrsJzqtbQgKO2UhOuhnO2GoO2DgOyehSDshKTqs4QpICjsmKTripjsl4XrrLRfMjAyNi0wNi0yMy/smKTripjruIzrpqztlZEubWQpIC8g7JiB7IiZOiA6IOy0iOq4sCDsnKDsnoUg7Yq4656Y7ZS9IO2ZleuztCDrsI8g7L2Y7YWQ7LigIOq4sO2ajSAo7YyM7J207ZSE65287J24IOyZhOyEsSkgKOyYpOuKmOyXheustF8yMDI2LTA2LTIzL+y1nOyiheu4jOumrO2VkS5tZCkifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi67O0656P67mbIOq0gOugqO2VtOyEnCDshKTrqoXtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IiMg67O0656P67mbIOyGjOqwgCDsmKjri6QgKFB1cnBsZSBDb3cpXG4jIOuztOuej+u5myDshozqsIAg7Jio64ukIChQdXJwbGUgQ293KSDigJQg66eI7LyA7YyFIOyZhOyghCDsoJXrpqxcblxu7IS47IqkIOqzoOuUmChTZXRoIEdvZGluKeydmCDrp4jsvIDtjIUg6rOg7KCELiDtlZwg66y47J6lOiBcIu2PieuylO2VmOuptCDrs7TsnbTsp4Ag7JWK64qU64ukLiDso7zrqqntlaAg66eM7ZWY6rKMKHJlbWFya2FibGUpIOunjOuTpOyWtOudvC5cIlxuXG4jIyAxLiDrs7Trno/ruZsg7IaMID0g66as66eI7Luk67iUXG4tIO2PieuylO2VnCDshowg7IiY67CxIOuniOumrOuKlCDslYgg67O07J2464ukLiDrs7Trno/ruZsg7IaM64qUIOuIhOq1rOuCmCDrqYjstrDshJwg67O06rOgIOy5nOq1rOyXkOqyjCDrp5DtlZzri6QuXG4tIOumrOuniOy7pOu4lCA9IFwicmVtYXJrKOyWuOq4iSntlaAg66eM7ZWcXCIuIOuniOy8gO2MheydmCDtlbXsi6zsnYAg6rSR6rOgIOq4sOyIoOydtCDslYTri4jrnbwg7KCc7ZKIIOyekOyytOqwgCDso7zrqqntlaAg66eM7ZWc6rCA64ukLlxuLSDrpqzrp4jsu6TruJTsnZgg67CY64yA66eQ7J2AIFwi64KY7IGoXCLsnbQg7JWE64uI6528IFwi66ek7JqwIOyii+ydjCh2ZXJ5IGdvb2QpXCIuIOustOuCnO2VmOqzoCDslYjsoITtlZwg6rKD7J2AIOuztOydtOyngCDslYrripTri6QuXG5cbiMjIDIuIOyYmyDrp4jsvIDtjIXsnZgg7KO97J2MIOKAlCBUVsK37IKw7JeFIOuzte2VqeyytFxuLSDtj4nrspTtlZwg7KCc7ZKIICsg66eJ64yA7ZWcIOq0keqzoOu5hCA9IOunpOy2nCwg7J20IOqzteyLneydtCDrrLTrhIjsoYzri6QuXG4tIOyCrOuejOuTpOydgCDqtJHqs6Drpbwg66y07Iuc7ZWY64qUIOuyleydhCDrsLDsm6Dri6QuIOq0keqzoOuhnCDtj4nrspTtlZwg7KCc7ZKI7J2EIOq1rO2VoCDsiJgg7JeG64ukLlxuLSDrp4jsvIDtjIXsnYQg7KCc7ZKIIOuBneyXkCDrjafrtpnsnbTsp4Ag66eQ6rOgLCDsoJztkogg7JWI7JeQIOuCtOyepe2VmOudvC5cblxuIyMgMy4g7IOI66Gc7Jq0IFAg4oCUIFB1cnBsZSBDb3dcbi0g7KCE7Ya1IOuniOy8gO2MhSBQKFByb2R1Y3QvUHJpY2UvUHJvbW90aW9uL1Bvc2l0aW9uaW5n4oCmKeyXkCAnUHVycGxlIENvdyfrpbwg642U7ZWY6528LlxuLSDquLDtmo0g7LKrIOuLqOqzhOu2gO2EsCBcIuydtOqyjCDsmZwg7J6F7IaM66y4IOuCoOq5jD9cIuulvCDshKTqs4Tsl5Ag64Sj7Ja06528LlxuXG4jIyA0LiDriITqtazsl5Dqsowg4oCUIOyYpO2DgOy/oChPdGFrdSlcbi0g66qo65GQ66W8IOunjOyhseyLnO2CpOugpOuKlCDsoJztkojsnYAg7JWE66y064+EIOyjvOuqqe2VmOyngCDslYrripTri6QuXG4tIOyYpO2DgOy/oCA9IOyWtOuWpCDqsoPsl5Ag67mE7KCV7IOB7KCB7Jy866GcIOyXtOq0ke2VmOuKlCDshozsiJguIOuPiMK37Iuc6rCE7J2EIOyTsOqzoCDrgqjsl5Dqsowg65ag65Og64ukLlxuLSDrr7jsp4Dqt7ztlZwg64uk7IiY67O064ukIOyXtOq0ke2VmOuKlCDshozsiJjrpbwg64W466Ck6528LiDqt7jrk6TsnbQg7Iuc7J6l7J2EIOyXsOuLpC5cblxuIyMgNS4g7Ja065a76rKMIO2NvOyngOuCmCDigJQg7Iqk64uI7KCAKFNuZWV6ZXIp7JmAIOyVhOydtOuUlOyWtCDrsJTsnbTrn6zsiqRcbi0g7Iqk64uI7KCAID0g7JWE7J2065SU7Ja066W8IOyerOyxhOq4sO2VmOuTryDtjbzrnKjrpqzripQg7KCE7YyM7J6QLiDsnoXshozrrLjsnZgg7ZW17IusLlxuLSDrpqzrp4jsu6TruJTtlZwg6rKD66eMIOyghO2MjOuQnOuLpC4g7Y+J67KU7ZWcIOqxtCDslYTrrLTrj4Qg7Lmc6rWs7JeQ6rKMIOunkO2VmOyngCDslYrripTri6QuXG4tIOq0keqzoOu5hCDrjIDsi6AsIOyKpOuLiOyggOqwgCDsnpDrsJzsoIHsnLzroZwg7Y2865yo66a0IOydtOycoOulvCDsoJztkojsl5Ag7Ius7Ja06528LiDtjbzrnKjrpqzquLAg7Im96rKMIOunjOuTpOyWtOudvC5cblxuIyMgNi4g7Ja866as7Ja064u17YSw7JmAIOy6kOymmFxuLSDrjIDspJHsnYQg7KeB7KCRIOuFuOumrOyngCDrp4jrnbwuIOyWvOumrOyWtOuLte2EsOulvCDrhbjrpqzqs6Ag6re465Ok7J20IOuLpOyImOyXkOqyjCDsoITtjIztlZjqsowg7ZWY6528LlxuLSDrpqzrp4jsu6TruJTtlZjsp4Ag7JWK7Jy866m0IOyWvOumrOyWtOuLte2EsOyZgCDri6TsiJgg7IKs7J20IOy6kOymmCjqsITqt7kp7J2EIOuquyDrhJjqs6Ag7IKs65287KeE64ukLlxuXG4jIyA3LiDqt7nri6jsnLzroZwsIOqwgOyepeyekOumrOuhnFxuLSDsi5zsnqUg7ZWc6rCA7Jq0642wKOykkeqwhCDtkojsp4jCt+qwgOqyqcK366y064Kc7ZWoKeuKlCDqsIDsnqUg67aQ67mE6rOgIOqwgOyepSDslYgg67O07J2464ukLlxuLSDtlZwg7LaV7JeQ7IScIOq3ueuLqOycvOuhnDog6rCA7J6lIOu5oOuluC/si7wv6rOg6riJL+uLqOyInO2VnC/soITrrLjsoIHsnbguIOyWtOykkeqwhO2VqOydtCDqsIDsnqUg7JyE7ZeY7ZWY64ukLlxuXG4jIyA4LiDrkZDroKTsm4DsnbQg7Y+J67KU7ZWo7J2EIOunjOuToOuLpFxuLSDrpqzrp4jsu6TruJTtlZjsp4Ag66q77ZWY64qUIOydtOycoOuKlCDruYTtjJDCt+yLpO2MqOqwgCDrkZDroKTsm4zshJwuIOyViOyghCJ9XX0KeyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiIxMDDsnbQo6rCAKSDrrZTsp4Ag7JWM66Ck7KSE656YPyJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiIjIE1yQmVhc3Qg7ZuE7YK5IOuhnOyngVxuIyBNckJlYXN0IO2bhO2CuSDroZzsp4Eg67aE7ISdXG5cbiMjIO2VteyLrCDtjKjthLRcbi0gKirssqsgNey0iCoqOiDstqnqsqnsoIEg7ZaJ64+ZwrfqsrDqs7wg66+466as67O06riwIChcIuyasOumrOuKlCDsnbQg7IKs656M7JeQ6rKMIDEwMOunjCDri6zrn6zrpbwg7KSs7Ja07JqULi4uXCIpXG4tICoqNX4zMOy0iCoqOiDsnITquLAg7ISk7KCVwrfsnbTtlbTqtIDqs4Qg66qF7IucIChcIi4uLu2VmOyngOunjCDsobDqsbTsnbQg7J6I7KOgLlwiKVxuLSAqKuqzoOuwgOuPhCDsu7cqKjog7Y+J6regIDEuNey0iOuLuSAx7Lu3LCDsi5zshKAg66q7IOuWvOqyjFxuLSAqKuyIq+yekCDqsJXsobAqKjog7ZWt7IOBIOq1rOyytOyggSDsiJjsuZggKFwiMTAw66eMIOuLrOufrFwiLCBcIjI07Iuc6rCEXCIsIFwiN+uqhVwiKVxuXG4jIyDsoIHsmqkg7LK07YGs66as7Iqk7Yq4XG4tIFsgXSDssqsgNey0iOyXkCDqsrDqs7wg66+466as67O06riwIOyeiOuCmD9cbi0gWyBdIOyLnOyyreyekOqwgCBcIuydtOqyjCDsp4Tsp5w/XCIg7J2Y7Ius7ZWY6rKMIOunjOuTnOuCmD9cbi0gWyBdIDMw7LSIIOyViOyXkCDsnITquLDCt+ydtO2VtOq0gOqzhCDrqoXtmZXtlZzqsIA/XG4tIFsgXSDsu7cg7Y+J6regIOq4uOydtOqwgCAy7LSIIOydtO2VmOyduOqwgD8ifV19CnsiY29udmVyc2F0aW9ucyI6W3sicm9sZSI6InVzZXIiLCJjb250ZW50Ijoi7Iqk7YKs7JeQIOuMgO2VtCDrhKTqsIAg7JWE64qUIOqxuCDrp5DtlbTspJguIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IiMg7Iqk7YKsOiDwn46sIO2bhO2CuSDrtoTshJ3quLBcbuuzuOyduCDssYTrhJAg7LWc6re8IOyYgeyDgeydmCDssqsgMzDstIgg7ZuE7YK5IO2MqO2EtOydhCDsnpDrj5kg67aE7ISdLlxu7Iuk7ZaJIOqwgOuKpe2VnCDtjIzsnbTsjawg7Iqk7YKsOiA8cnVuPnB5dGhvbjMgXCJDOlxcVXNlcnNcXGJzZzEyXFxEZXNrdG9wXFxjb25uZWN0LWFpLXBhY2tzXFzsiqTtgqxcXHlvdXR1YmVcXGhvb2tfYW5hbHl6ZXIucHlcIjwvcnVuPiJ9XX0="
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 40, learning_rate = 0.0005,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 모델/버전마다 다름(<|turn> vs <start_of_turn>) → 실제 텍스트에서 자동 감지
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
_im = "<|turn>user\n" if "<|turn>user" in _t else "<start_of_turn>user\n"
_rm = "<|turn>model\n" if "<|turn>model" in _t else "<start_of_turn>model\n"
trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 학습 준비 완료")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


## 💾 GGUF로 저장 (Connect AI 내장 엔진용)
테스트가 만족스러우면 업로드! (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
print("메모리 정리 완료 — GGUF 변환 시작")
# 내 모델 = 장기 기억. q4_k_m GGUF 로 저장 + HF 업로드
model.push_to_hub_gguf("byeon120/my-brain-v1", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ 완료! Connect AI 앱 → 🤖 내 AI 팀 → HuggingFace에서 받기 → \"byeon120/my-brain-v1\" 검색해서 내려받으면 내 모델로 바로 사용 (LM Studio 불필요)")
